In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
import time
import os

DATA_PATH = "/content/drive/MyDrive/medrag/pubmedqa_filtered.json"
OUTPUT_DIR = "/content/drive/MyDrive/medrag/biomistral_rag_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"Corpus size: {len(corpus)} samples")
print(f"Keys per sample: {list(corpus[0].keys())}")
print(f"\nSample query: {corpus[0]['query']}")
print(f"Sample abstract preview: {corpus[0]['supporting_abstracts'][0][:200]}")

Mounted at /content/drive
Corpus size: 759 samples
Keys per sample: ['pubid', 'query', 'gold_answer', 'supporting_abstracts', 'label', 'source']

Sample query: Landolt C and snellen e acuity: differences in strabismus amblyopia?
Sample abstract preview: Assessment of visual acuity depends on the optotypes used for measurement. The ability to recognize different optotypes differs even if their critical details appear under the same visual angle. Since


In [2]:
documents = []
doc_metadata = []

for sample in corpus:
    for abstract in sample["supporting_abstracts"]:
        documents.append(abstract)
        doc_metadata.append({
            "pubid": sample["pubid"],
            "query": sample["query"],
            "gold_answer": sample["gold_answer"],
            "label": sample["label"]
        })

print(f"Total documents indexed: {len(documents)}")
print(f"Avg abstracts per sample: {len(documents)/len(corpus):.2f}")

print("\nBuilding BM25 index...")
start = time.time()
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)
print(f"BM25 index built in {time.time()-start:.2f}s")
print(f"Corpus vocabulary size: {len(bm25.idf)}")

Total documents indexed: 2542
Avg abstracts per sample: 3.35

Building BM25 index...
BM25 index built in 0.08s
Corpus vocabulary size: 23619


In [3]:
def bm25_retrieve(query, k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_k_indices = np.argsort(scores)[::-1][:k]

    results = []
    for idx in top_k_indices:
        results.append({
            "abstract": documents[idx],
            "score": float(scores[idx]),
            "pubid": doc_metadata[idx]["pubid"],
            "gold_answer": doc_metadata[idx]["gold_answer"],
            "label": doc_metadata[idx]["label"]
        })
    return results

test_queries = [corpus[0]["query"], corpus[10]["query"], corpus[50]["query"]]

print("=== BM25 RETRIEVAL VALIDATION ===\n")
for query in test_queries:
    print(f"Query: {query}")
    results = bm25_retrieve(query, k=3)
    for i, r in enumerate(results):
        print(f"  [{i+1}] score={r['score']:.3f} | pubid={r['pubid']} | "
              f"abstract preview: {r['abstract'][:100]}")
    print()

=== BM25 RETRIEVAL VALIDATION ===

Query: Landolt C and snellen e acuity: differences in strabismus amblyopia?
  [1] score=38.876 | pubid=16418930 | abstract preview: Differences between Landolt C acuity (LR) and Snellen E acuity (SE) were small. The mean decimal val
  [2] score=28.594 | pubid=16418930 | abstract preview: Assessment of visual acuity depends on the optotypes used for measurement. The ability to recognize 
  [3] score=23.400 | pubid=16418930 | abstract preview: 100 patients (age 8 - 90 years, median 60.5 years) with various eye disorders, among them 39 with am

Query: Is there a model to teach and practice retroperitoneoscopic nephrectomy?
  [1] score=14.766 | pubid=22266735 | abstract preview: We developed a decision analysis model comparing the cost-utility of three strategies to identify GD
  [2] score=14.647 | pubid=12153648 | abstract preview: Women's experiences of childbirth may affect their future reproduction, and the model of care affect
  [3] score=14.439 | pu

In [5]:
from sentence_transformers import SentenceTransformer

print("Loading biomedical sentence encoder...")
enc_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
print("Encoder loaded.")

print(f"\nEmbedding {len(documents)} documents...")
start = time.time()
doc_embeddings = enc_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"\nDone in {time.time()-start:.1f}s")
print(f"Embedding matrix shape: {doc_embeddings.shape}")

faiss.normalize_L2(doc_embeddings)
dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)
print(f"FAISS index built. Total vectors: {index.ntotal}")

Loading biomedical sentence encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoder loaded.

Embedding 2542 documents...


Batches:   0%|          | 0/80 [00:00<?, ?it/s]


Done in 4.5s
Embedding matrix shape: (2542, 768)
FAISS index built. Total vectors: 2542


In [6]:
def faiss_retrieve(query, k=3):
    query_embedding = enc_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "abstract": documents[idx],
            "score": float(score),
            "pubid": doc_metadata[idx]["pubid"],
            "gold_answer": doc_metadata[idx]["gold_answer"],
            "label": doc_metadata[idx]["label"]
        })
    return results

test_queries = [corpus[0]["query"], corpus[10]["query"], corpus[50]["query"]]

print("=== BM25 vs FAISS RETRIEVAL COMPARISON ===\n")
for query in test_queries:
    print(f"Query: {query}")

    bm25_results = bm25_retrieve(query, k=3)
    faiss_results = faiss_retrieve(query, k=3)

    print(f"  BM25  [1] score={bm25_results[0]['score']:.3f} | pubid={bm25_results[0]['pubid']} | {bm25_results[0]['abstract'][:80]}")
    print(f"  FAISS [1] score={faiss_results[0]['score']:.3f} | pubid={faiss_results[0]['pubid']} | {faiss_results[0]['abstract'][:80]}")
    print()

=== BM25 vs FAISS RETRIEVAL COMPARISON ===

Query: Landolt C and snellen e acuity: differences in strabismus amblyopia?
  BM25  [1] score=38.876 | pubid=16418930 | Differences between Landolt C acuity (LR) and Snellen E acuity (SE) were small. 
  FAISS [1] score=0.825 | pubid=16418930 | Differences between Landolt C acuity (LR) and Snellen E acuity (SE) were small. 

Query: Is there a model to teach and practice retroperitoneoscopic nephrectomy?
  BM25  [1] score=14.766 | pubid=22266735 | We developed a decision analysis model comparing the cost-utility of three strat
  FAISS [1] score=0.617 | pubid=22694248 | Although the retroperitoneal approach has been the preferred choice for open uro

Query: Does the clinical presentation of a prior preterm birth predict risk in a subsequent pregnancy?
  BM25  [1] score=43.458 | pubid=26215326 | The objective of the study was to determine whether risk of recurrent preterm bi
  FAISS [1] score=0.769 | pubid=26215326 | The objective of the study wa

In [7]:
def hybrid_retrieve(query, k=3, rrf_k=60):
    bm25_results = bm25_retrieve(query, k=k*2)
    faiss_results = faiss_retrieve(query, k=k*2)
    combined = {}

    for rank, r in enumerate(bm25_results):
        key = r["abstract"]
        combined[key] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}

    for rank, r in enumerate(faiss_results):
        key = r["abstract"]
        if key in combined:
            combined[key]["score"] += 1 / (rrf_k + rank + 1)
        else:
            combined[key] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}

    sorted_results = sorted(combined.values(), key=lambda x: x["score"], reverse=True)[:k]
    return [r["meta"] for r in sorted_results]

def build_rag_prompt(query, k=3, tokenizer=None):
    results = hybrid_retrieve(query, k=k)

    context_parts = [f"[{i+1}] {r['abstract']}" for i, r in enumerate(results)]
    context = "\n\n".join(context_parts)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

    span_index = None
    if tokenizer is not None:
        span_index = []
        tokens_so_far = tokenizer("Context:\n", return_tensors="pt")["input_ids"].shape[1]

        for i, part in enumerate(context_parts):
            part_len = tokenizer(part, return_tensors="pt")["input_ids"].shape[1]
            sep_len = tokenizer("\n\n", return_tensors="pt")["input_ids"].shape[1]

            span_index.append({
                "doc_rank": i,
                "pubid": results[i]["pubid"],
                "token_start": tokens_so_far,
                "token_end": tokens_so_far + part_len
            })

            tokens_so_far += part_len + sep_len

    return prompt, results, span_index

def build_suppressed_prompt(query):
    prompt = f"Question: {query}\nAnswer:"
    return prompt

print("Retrieval and prompt functions ready.")

Retrieval and prompt functions ready.


In [8]:
print("=== HYBRID RETRIEVAL VALIDATION ===\n")

validation_log = []

for idx in [0, 10, 50]:
    sample = corpus[idx]
    query = sample["query"]

    prompt, results, span_index = build_rag_prompt(query, k=3)

    validation_log.append({
        "idx": idx,
        "pubid": sample["pubid"],
        "query": query,
        "label": sample["label"],
        "top_pubids": [r["pubid"] for r in results],
        "top_scores": [float(r["score"]) for r in results],
        "prompt_chars": len(prompt)
    })

    print(f"Query: {query}")
    for i, r in enumerate(results):
        print(f"  [{i+1}] pubid={r['pubid']} | score={r['score']:.4f} | {r['abstract'][:100]}")
    print(f"Prompt preview:\n{prompt[:500]}")
    print()

=== HYBRID RETRIEVAL VALIDATION ===

Query: Landolt C and snellen e acuity: differences in strabismus amblyopia?
  [1] pubid=16418930 | score=38.8760 | Differences between Landolt C acuity (LR) and Snellen E acuity (SE) were small. The mean decimal val
  [2] pubid=16418930 | score=28.5940 | Assessment of visual acuity depends on the optotypes used for measurement. The ability to recognize 
  [3] pubid=16418930 | score=23.4000 | 100 patients (age 8 - 90 years, median 60.5 years) with various eye disorders, among them 39 with am
Prompt preview:
Context:
[1] Differences between Landolt C acuity (LR) and Snellen E acuity (SE) were small. The mean decimal values for LR and SE were 0.25 and 0.29 in the entire group and 0.14 and 0.16 for the eyes with strabismus amblyopia. The mean difference between LR and SE was 0.55 lines in the entire group and 0.55 lines for the eyes with strabismus amblyopia, with higher values of SE in both groups. The results of the other groups were similar with only

In [9]:
report_path = os.path.join(OUTPUT_DIR, "rag_validation_report.json")

with open(report_path, "w") as f:
    json.dump({
        "n_corpus_samples": len(corpus),
        "n_documents": len(documents),
        "retriever": "hybrid_bm25_faiss_rrf",
        "sentence_encoder": "NeuML/pubmedbert-base-embeddings",
        "validation_samples": validation_log
    }, f, indent=2)

print(f"Validation report saved to {report_path}")
print("01_biomistral_rag_pipeline complete")

Validation report saved to /content/drive/MyDrive/medrag/biomistral_rag_outputs/rag_validation_report.json
01_biomistral_rag_pipeline complete
